# Clark, Abbott & Sompolinsky (2026): the data-derived network, on DANDI:000939

Paper. doi:10.1038/s41593-026-02452-4; claims file `claims/clark-2026.yaml`, written from the
version of record and its supplement.

Dataset. DANDI:000939 at version 0.240528.1542, the version the paper cites; every file
was replaced in later versions (see `tuning_curves.ipynb`, "Files and versions").

The tuning curves are computed from the streamed NWB files as in `tuning_curves.ipynb`
(`hdring.load_all_curves`). The network is built and run as in the paper's
`notebooks/data_driven_attractor.ipynb`, `spurious_fixed_points.ipynb` and `ring/data_driven.py`
(commit 245726f), in numpy and float32 (the paper's simulations use float32 torch tensors).
The lambda and c grid of Supplementary Figs. 8 and 9 is in `network_grid.ipynb`; the
simulations of the generative-process and rank-four networks are in `model_sims.ipynb`.

| claim | tested here |
|---|---|
| fig2b-sorted-weights-mexican-hat | yes |
| fig2ef-data-network-quasicontinuous | yes |
| fig2gi-circulant-noise-collapses | yes |
| fig2jk-velocity-integration | yes |
| figs14-data-spectrum-doublets | yes |
| figs11b-data-network-global-stability | yes |
| figs8-s9-lambda-and-c | in network_grid.ipynb |

In [ ]:
import os, time, resource
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.interpolate import interp1d
from astropy.stats import circcorrcoef
import hdring as H

T0 = time.time()
print("dandiset", H.DANDISET, "version", H.VERSION)

In [ ]:
sessions, Phi_data, mouse_nums, names = H.load_all_curves()
print(f"loaded in {time.time()-T0:.0f} s")

In [ ]:
print("sessions", len(sessions), "| units", sum(d["sess"]["n_units"] for d in sessions),
      "| head-direction units", sum(int(d["sess"]["is_hd"].sum()) for d in sessions), "| curves used", Phi_data.shape)
print("curve values: min", Phi_data.min().round(5), "max", Phi_data.max().round(3), "| row means 1:", np.allclose(Phi_data.mean(1), 1))
print("cells per mouse:", dict(zip(names, np.bincount(mouse_nums))))

## The network

Supplementary Note 3, section 8.1, as `prepare_smoothed_normalized_resampled_tuning_curves` and
`compute_weights` (lambda 1e-8). The paper's notebook prints the mean squared reconstruction
error, `err = 1.7152363818838536e-08`; the same number from these data is printed below.

In [ ]:
t = time.time()
X, Phi = H.prepare_network_curves(Phi_data)
J, J_prime = H.compute_weights(X, Phi, lamda=1e-8)
err = np.mean((X - Phi @ J.T) ** 2)
N = J.shape[0]
print(f"X, Phi {X.shape}; J {J.shape}; built in {time.time()-t:.0f} s")
print(f"err = {err!r} (paper's notebook: 1.7152363818838536e-08)")
print("diagonal of J is zero:", np.allclose(np.diag(J), 0), "| J range", J.min().round(4), J.max().round(4))
print("Sinkhorn result: mean over neurons of Phi at each angle, range", Phi.mean(1).min().round(4), Phi.mean(1).max().round(4))
Phi32 = Phi.astype(np.float32); Phi_sq = (Phi32 ** 2).sum(1)

## fig2b-sorted-weights-mexican-hat

> The optimized data-derived weight matrix looks unstructured in arbitrary order, and ordering
> neurons by their tuning-curve centers of mass reveals a noisy approximate Mexican hat
> structure.

In [ ]:
com_idx = np.floor(H.circular_center_of_mass(Phi.T) * Phi.shape[0] / (2 * np.pi)).astype(int)   # data_driven.circular_com(Phi.T, True)
order = np.argsort(com_idx)
J_srt = J[order][:, order]
prof = H.circularized_profile(J_srt)
prof_arb = H.circularized_profile(J)
offs = (np.arange(N) + N // 2) % N - N // 2
near = (np.abs(offs) <= N // 20) & (offs != 0)
mid = (np.abs(offs) >= N // 8) & (np.abs(offs) <= 3 * N // 8)
print(f"sorted profile: maximum {prof.max():.5f} at offset {offs[prof.argmax()]} (within N/20 = {N//20}: {abs(offs[prof.argmax()]) <= N//20})")
print(f"mean over offsets {prof.mean():.6f}; minimum over intermediate offsets (N/8 to 3N/8) {prof[mid].min():.6f} at offset {offs[mid][prof[mid].argmin()]}")
print(f"fraction of intermediate offsets below the mean {np.mean(prof[mid] < prof.mean()):.2f}; below 0 {np.mean(prof[mid] < 0):.2f}")
print(f"near offsets (1 to N/20) mean {prof[near].mean():.6f} (positive: local excitation)")
print(f"arbitrary order: profile range {prof_arb[offs != 0].min():.6f} to {prof_arb.max():.6f}, sd {prof_arb[offs != 0].std():.6f}; sorted profile sd {prof[offs != 0].std():.6f}")
print("sorted profile at offsets 0, +-10, +-50, +-100, +-200, +-400, +-766:", {o: round(float(prof[o % N]), 6) for o in [0, 10, -10, 50, -50, 100, -100, 200, -200, 400, -400, 766]})
rows_sorted_signal = np.mean([np.corrcoef(np.roll(J_srt[i], -i), prof)[0, 1] for i in range(N)])
print(f"mean correlation of each sorted row (aligned) with the profile: {rows_sorted_signal:.3f}; same for arbitrary order: {np.mean([np.corrcoef(np.roll(J[i], -i), prof_arb)[0, 1] for i in range(N)]):.3f}")
fig, ax = plt.subplots(1, 3, figsize=(9, 2.6))
ax[0].imshow(J, vmin=-0.005, vmax=0.005, cmap="RdBu_r"); ax[0].set_title("J, arbitrary order")
ax[1].imshow(J_srt, vmin=-0.005, vmax=0.005, cmap="RdBu_r"); ax[1].set_title("J, sorted by center of mass")
ax[2].plot(offs[np.argsort(offs)], prof[np.argsort(offs)]); ax[2].axhline(0, c="k", lw=0.5); ax[2].set_title("diagonal profile"); ax[2].set_xlabel("offset (neurons)")
plt.show()
print("colour limits +-0.005; entries of J outside them:", int((np.abs(J) > 0.005).sum()), "of", J.size)

**Result: reproduced.** Sorted by center of mass, the mean of the weights along each circular
diagonal peaks at offset -8 (0.0053), stays positive over offsets up to N/20 (mean 0.0040), and
falls below its mean at 88% of intermediate offsets and below zero at 80%, down to -0.00099 at
offset 571: local excitation and longer-range inhibition. In arbitrary order the same profile is
flat (sd 0.00015 against 0.00147 sorted), and an aligned sorted row correlates with the profile at
0.356 on average against 0.026 in arbitrary order.

## fig2ef-data-network-quasicontinuous

> Started at regularly spaced manifold points perturbed by noise, the data-derived network's
> trajectories return to the manifold within about 1 s and then persist over 30 s, an
> order-of-magnitude separation between convergence and drift timescales.

`run_convergence_analysis`: T 30 s, tau 50 ms, step 1 ms, c 1; 100 initial conditions at every
100th target angle with Gaussian noise of standard deviation 3 on the input currents. The
distance to the manifold (nearest of the 10,000 target firing-rate points, over sqrt(N)) and the
manifold angle (angle of that point) are taken every 100 ms; the paper's notebook saves every
10 ms, a plotting detail that cannot change the result at the times the criterion reads.

In [ ]:
def run_2ef(sd):
    rng = np.random.default_rng(sd)
    x0 = X[::100].T + rng.standard_normal((N, 100)) * 3.0
    rec_t, rec_d, rec_i = [], [], []
    def on_save(tt, x):
        idx, dist = H.nearest_target(Phi32, H.nonlin(x), Phi_sq)
        rec_t.append(tt); rec_d.append(dist); rec_i.append(idx)
    xf = H.simulate(J, x0, T=30.0, dt=0.001, c=1.0, save_every=0.1, on_save=on_save)
    return np.array(rec_t), np.array(rec_d), np.array(rec_i), xf
def judge_2ef(tt, dd, ii):
    k0, k1, k20 = 0, np.argmin(np.abs(tt - 1)), np.argmin(np.abs(tt - 20))
    m0, m1, m20 = np.median(dd[k0]), np.median(dd[k1]), np.median(dd[k20])
    ang = 2 * np.pi * ii / X.shape[0]
    move = H.circ_diff(ang[-1], ang[k1])
    nbins = len(np.unique(np.floor(ang[-1] / (2 * np.pi / 100)).astype(int)))
    return dict(med0=m0, med1=m1, med20=m20, a=m1 <= m0 / 10, b=m1 <= 2 * m20,
                c_frac=np.mean(move < 2 * np.pi / 20), c=np.mean(move < 2 * np.pi / 20) >= 0.9, d_bins=nbins, d=nbins >= 50,
                t_last=tt[-1], med_last=np.median(dd[-1]), max20=dd[k20].max(), move_med=np.median(move))
res2ef = {}
t = time.time()
for k in range(5):
    sd = H.seed("fig2ef-data-network-quasicontinuous") + 1000 * k
    tt, dd, ii, xf = run_2ef(sd)
    res2ef[sd] = (tt, dd, ii)
    j = judge_2ef(tt, dd, ii)
    print(f"seed {sd}: median distance at 0 s {j['med0']:.4f}, 1 s {j['med1']:.5f}, 20 s {j['med20']:.5f}, {j['t_last']:.2f} s {j['med_last']:.5f}; "
          f"(a) {j['a']} (b) {j['b']}; angle moved under 2pi/20 between 1 s and the end: {j['c_frac']:.2f} (c) {j['c']}; "
          f"final angles in {j['d_bins']} of 100 bins (d) {j['d']}; median movement {j['move_med']:.4f} rad")
    if k == 0:
        sd0 = sd
        print("   median distance over time (s):", {round(float(tt[q]), 1): round(float(np.median(dd[q])), 5) for q in [0, 1, 2, 3, 5, 10, 20, 50, 100, 200, len(tt) - 1] if q < len(tt)})
print(f"time {time.time()-t:.0f} s")
tt, dd, ii = res2ef[sd0]
fig, ax = plt.subplots(1, 2, figsize=(8, 2.4))
ax[0].semilogy(tt, dd, lw=0.3); ax[0].set_xlabel("time (s)"); ax[0].set_ylabel("distance / sqrt(N)")
ax[1].plot(tt, np.unwrap(2 * np.pi * ii / X.shape[0], axis=0), lw=0.3); ax[1].set_xlabel("time (s)"); ax[1].set_ylabel("manifold angle")
plt.show()
print("distance range", dd.min(), dd.max())
D20_MAX = res2ef[sd0][1][np.argmin(np.abs(tt - 20))].max(); D20_MED = np.median(res2ef[sd0][1][np.argmin(np.abs(tt - 20))])

**Result: reproduced.** The median distance to the manifold falls from 1.966 at 0 s to 0.0816 at 1
s, under one tenth (reading a), and to 0.00068 at 20 s; reading (b) fails, as the distance keeps
falling after 1 s. Between 1 s and the end, every trajectory's manifold angle moves by less than 2
pi / 20 (median 0.0025 rad, reading c), and the 100 final angles occupy 77 of 100 bins (reading
d). The outcome is the same under all five seeds.

## fig2gi-circulant-noise-collapses

> The optimal weights keep a continuum of stable heterogeneous states; their circulant average
> gives bumps identical up to translation, while the circulant matrix plus the scrambled
> residuals collapses onto a few discrete states within 30 s, so the optimal weights are not a
> noisy circulant matrix.

As `compute_new_matrices` and `run_all_simulations`: the sorted matrix; its circulant average;
the circulant plus the residuals shuffled over all off-diagonal entries (asymmetric version,
Fig. 2d,i), and the symmetric version as a variant; both surrogates times 1.275. 100 initial
conditions at regularly spaced points of the sorted targets, noise of standard deviation 2.5,
c 1, 30 s. The final position is the peak of the final input currents smoothed across sorted
neurons with a Gaussian of 2.5 neurons (`plot_simulation_results`, gaussian_filter1d default
mode). The claims file merges positions within 31 neurons into one state; it does not say how,
so three readings are counted (helper, `count_states`): single linkage, greedy, and occupied
bins of 31 neurons. The neuron at the peak of the smoothed currents is often the same for nearby
angles (the initial states themselves give only 24 to 33 states under these readings), so the
50 states the criterion asks of the optimal matrix are out of reach of this position measure.
As a variant, the position is taken as the manifold angle of the nearest target point, which can
give up to 50 bins of 2 pi / 50.

In [ ]:
X_srt = X[:, order]
J_circ = np.asarray(H.circulant_projection(J_srt))
def scrambled(rng):
    Jn = J_circ - J_srt
    iu = np.triu_indices(N, k=1)
    z = np.concatenate((Jn[iu], Jn.T[iu])); z = rng.permutation(z)
    A = np.zeros_like(Jn); A[iu] = z[:len(z) // 2]; A.T[iu] = z[len(z) // 2:]
    S = np.zeros_like(Jn); S[iu] = z[:len(z) // 2]; S.T[iu] = z[:len(z) // 2]
    return A, S
def run_final(M, sd, return_x0=False):
    rng = np.random.default_rng(sd)
    x0 = X_srt[::int(X_srt.shape[0] / 100)].T + rng.standard_normal((N, 100)) * 2.5
    xf = H.simulate(M, x0, T=30.0, dt=0.001, c=1.0)
    return (xf, x0) if return_x0 else xf
def positions(xf, mode="reflect"):
    # data_driven_attractor.ipynb smooths with gaussian_filter1d's default mode (reflect), as here
    return gaussian_filter1d(xf.astype(np.float64), sigma=2.5, axis=0, mode=mode).argmax(0)
def shift_corr(xf):
    ref = xf[:, 0]
    best = []
    for b in range(1, xf.shape[1]):
        v = xf[:, b]
        F = np.fft.ifft(np.fft.fft(ref - ref.mean()) * np.conj(np.fft.fft(v - v.mean()))).real
        best.append(F.max() / (np.linalg.norm(ref - ref.mean()) * np.linalg.norm(v - v.mean())))
    return np.array(best)
W = 31
sd0 = H.seed("fig2gi-circulant-noise-collapses")
t = time.time()
res_gi = {}
A_noise, S_noise = scrambled(np.random.default_rng(sd0))
mats = {"sorted": J_srt, "circulant": 1.275 * J_circ, "circ_plus_noise": 1.275 * (J_circ + A_noise), "circ_plus_noise_sym": 1.275 * (J_circ + S_noise)}
for key, M in mats.items():
    xf, x0_gi = run_final(M, sd0 + 1, return_x0=True)   # second generator of the claim: initial-condition noise
    pos = positions(xf)
    res_gi[key] = (xf, pos)
    cs = H.count_states(pos, N, W)
    extra = ""
    if key == "circulant":
        sc = shift_corr(xf); extra = f"; correlation with the best shift of the first state: min {sc.min():.4f}, median {np.median(sc):.4f}"
    cs_wrap = H.count_states(positions(xf, "wrap"), N, W)
    print(f"{key}: states (smoothing mode wrap, variant) single {cs_wrap['single']}, greedy {cs_wrap['greedy']}, bins {cs_wrap['bins']}")
    print(f"{key}: states single {cs['single']}, greedy {cs['greedy']}, bins {cs['bins']}; distinct peak neurons {len(np.unique(pos))}; peak height range {xf.max(0).min():.2f} to {xf.max(0).max():.2f}{extra}")
print(f"time {time.time()-t:.0f} s")
Phi_srt32 = Phi32[:, order]; Phi_srt_sq = (Phi_srt32 ** 2).sum(1)
for key, (xf, pos) in res_gi.items():
    idx, dist = H.nearest_target(Phi_srt32, H.nonlin(xf), Phi_srt_sq)
    cs = H.count_states(idx, X.shape[0], X.shape[0] / 50)
    print(f"variant, position as the manifold angle of the nearest target point: {key}: states single {cs['single']}, greedy {cs['greedy']}, bins of 2 pi / 50 {cs['bins']}; median distance to the manifold {np.median(dist):.4f}")
sc_circ = shift_corr(res_gi["circulant"][0])
print("circulant states are shifts (all >= 0.99):", bool((sc_circ >= 0.99).all()))
print("reference: initial positions, states", H.count_states(positions(X_srt[::int(X_srt.shape[0] / 100)].T), N, W))
# the continuum tested directly: the manifold angle (nearest target point) of each sorted-matrix
# trajectory at the start (noisy initial state) and at the end
Phi_srt32_ = Phi32[:, order]
i0, _ = H.nearest_target(Phi_srt32_, H.nonlin(x0_gi.astype(np.float32)), Phi_srt_sq)
i1, d1 = H.nearest_target(Phi_srt32_, H.nonlin(res_gi["sorted"][0]), Phi_srt_sq)
shift = H.circ_diff(2 * np.pi * i1 / X.shape[0], 2 * np.pi * i0 / X.shape[0])
print(f"sorted matrix, initial-to-final manifold angle shift: median {np.median(shift):.4f} rad, 90th percentile {np.percentile(shift, 90):.4f}, max {shift.max():.4f}; "
      f"under 2 pi / 50 for {np.mean(shift < 2 * np.pi / 50):.2f} of trajectories; final angles in {len(np.unique(np.floor(i1 / (X.shape[0] / 50))))} of 50 bins")
# seed check on the random parts: the shuffle and initial noise of the circulant plus noise matrix,
# and the initial noise of the sorted matrix
for k in range(1, 5):
    sd = sd0 + 1000 * k
    A_k, _ = scrambled(np.random.default_rng(sd))
    pos = positions(run_final(1.275 * (J_circ + A_k), sd + 1))
    xs_k, x0_k = run_final(J_srt, sd + 1, return_x0=True)
    a0, _ = H.nearest_target(Phi_srt32_, H.nonlin(x0_k.astype(np.float32)), Phi_srt_sq)
    a1, _ = H.nearest_target(Phi_srt32_, H.nonlin(xs_k), Phi_srt_sq)
    sh = H.circ_diff(2 * np.pi * a1 / X.shape[0], 2 * np.pi * a0 / X.shape[0])
    print(f"seed {sd}: circ_plus_noise states", H.count_states(pos, N, W), "| sorted states", H.count_states(positions(xs_k), N, W),
          f"| sorted, manifold-angle shift median {np.median(sh):.4f} rad, max {sh.max():.4f}, final angles in {len(np.unique(np.floor(a1 / (X.shape[0] / 50))))} of 50 bins", flush=True)
print(f"time {time.time()-t:.0f} s")
fig, ax = plt.subplots(4, 1, figsize=(7, 5))
for a_, (key, (xf, pos)) in zip(ax, res_gi.items()):
    a_.imshow(gaussian_filter1d(xf, 2.5, axis=0), aspect="auto", vmin=0, vmax=6); a_.set_title(key, fontsize=8)
plt.tight_layout(); plt.show()
print("final input currents range over the four runs:", min(v[0].min() for v in res_gi.values()).round(2), max(v[0].max() for v in res_gi.values()).round(2), "(colour 0 to 6)")

**Result: partially reproduced.** The circulant plus scrambled residuals collapses to 4 states (2
to 7 over five seeds; the symmetric version to 2), and the circulant matrix's 100 final states are
exact shifts of one another (correlation 1.0000). The sorted optimal matrix keeps 24, 25 or 31
states under the three merging readings (24 to 26 and 31 to 32 over five seeds), short of the
criterion's 50; its initial states already give only 24 to 33 under this position measure, so the
50 cannot be reached by it (a question for a reader). Wrap-mode smoothing gives the same counts.
Read on the manifold angle, the sorted matrix keeps the continuum: each trajectory's angle moves
from its start by a median of 0.0050 rad (largest 0.0565, all under 2 pi / 50), the final angles
fill all 50 bins of 2 pi / 50 under all five seeds, at a median distance of 0.0008 from the
manifold, and the circulant plus noise fills 7. The contrast between the optimal matrix and the
circulant plus noise holds.

## fig2jk-velocity-integration

> With the velocity-coupling weights added, the data-derived network's overlap order parameter
> tracks a constant angular velocity input of 0.8 rad/s accurately and follows the more complex
> trajectory of the mouse's own head-direction behaviour.

As `run_analysis` in `data_driven_attractor.ipynb`: started at the target state of angle 0, c 1,
tau 50 ms, step 1 ms, 110 s; omega 0.8 rad/s from 20 s (j), or the angular velocity of A3707's
square-epoch head direction (`closing_the_loop.ipynb` saves `hd_full[ti_idx:tf_idx]` at 10 ms;
`preprocess_hd_data`: gaps interpolated on the circle, unwrapped, 2,000 still samples
prepended, Gaussian of 10 samples, differenced), linearly interpolated in time. Saved every 10
ms; the target is the cumulative sum of omega at the save times times 10 ms; the first 20 s
are dropped. The two inputs run as two columns of one simulation. The angle of the peak of
m = Phi r is compared with the target modulo 2 pi.

In [ ]:
d7 = sessions[int(np.where(names == "A3707")[0][0])]
s7 = d7["sess"]
ti_nom, tf_nom = [(a, b) for tg, a, b in s7["epochs"] if tg == "wake_square"][0]
ti_idx = np.argmin((ti_nom - s7["hd_t"]) ** 2); tf_idx = np.argmin((tf_nom - s7["hd_t"]) ** 2)
hd_ts = s7["hd"][ti_idx:tf_idx].copy()
def interpolate_angle_nans(hd):
    ok = np.where(~np.isnan(hd))[0]
    if len(ok) in (0, len(hd)): return hd
    zz = np.exp(1j * hd[ok]); ia = np.arange(len(hd))
    return np.mod(np.angle(np.interp(ia, ok, zz.real) + 1j * np.interp(ia, ok, zz.imag)), 2 * np.pi)
hu = np.unwrap(interpolate_angle_nans(hd_ts))
hu = np.concatenate((np.ones(2000) * hu[0], hu))
hs = gaussian_filter1d(hu, sigma=10)
omega_data = np.diff(np.unwrap(hs)) / 0.01
om_interp = interp1d(np.arange(len(omega_data)) * 0.01, omega_data)
print(f"A3707 square epoch: {len(hd_ts)} samples, {np.isnan(hd_ts).sum()} untracked; omega range {omega_data.min():.2f} to {omega_data.max():.2f} rad/s, median |omega| {np.median(np.abs(omega_data)):.3f}")
omega_fn = lambda tt: np.array([0.8 * (tt > 20.0), om_interp(tt)])
peaks, saves_t = [], []
def on_save_jk(tt, x):
    saves_t.append(tt); peaks.append((Phi32 @ H.nonlin(x)).argmax(0))
t = time.time()
H.simulate(J, np.repeat(X[0:1].T, 2, axis=1), T=110.0, dt=0.001, c=1.0, save_every=0.01, J_prime=J_prime, omega=omega_fn, on_save=on_save_jk)
print(f"simulated in {time.time()-t:.0f} s")
peaks = np.array(peaks); saves_t = np.array(saves_t)
ang = 2 * np.pi * peaks / X.shape[0]
skip = int(20 / 0.01)
for col, lab in ((0, "constant 0.8 rad/s"), (1, "A3707 behaviour")):
    om = np.array([omega_fn(tt)[col] for tt in saves_t])
    target = np.cumsum(om) * 0.01
    a_, t_ = ang[skip:, col], np.mod(target[skip:], 2 * np.pi)
    e = H.circ_diff(a_, t_)
    rc = circcorrcoef(a_, t_)
    off = np.angle(np.mean(np.exp(1j * (a_ - t_))))
    e_off = H.circ_diff(a_ - off, t_)
    print(f"{lab}: {len(a_)} samples from 20 to {saves_t[-1]:.2f} s; median |error| {np.median(e):.3f} rad (pi/8 {np.pi/8:.3f}), 90th percentile {np.percentile(e, 90):.3f}; circular r {rc:.3f}; "
          f"mean offset {off:.3f} rad, median |error| after removing it {np.median(e_off):.3f}; total turn of target {np.abs(target[-1]-target[skip]):.1f} rad")
    if col == 0:
        e_j, rc_j = np.median(e), rc
    else:
        e_k, rc_k = np.median(e), rc
print("bump angle at 0, 10 and 20 s (constant input column):", ang[[0, 1000, 2000], 0].round(3))
fig, ax = plt.subplots(2, 1, figsize=(8, 3))
for col in (0, 1):
    om = np.array([omega_fn(tt)[col] for tt in saves_t]); target = np.mod(np.cumsum(om) * 0.01, 2 * np.pi)
    ax[col].plot(saves_t[skip:], ang[skip:, col], ".", ms=0.5); ax[col].plot(saves_t[skip:], target[skip:], "r.", ms=0.3)
plt.show()

**Result: reproduced.** Over 20 to 110 s the angle of the peak of m follows the integrated
constant input of 0.8 rad/s with a median absolute error of 0.011 rad and circular correlation
1.000, and the integrated angular velocity of A3707 (13.2 rad of net turning) with a median error
of 0.013 rad and circular correlation 1.000, under both readings.

## figs14-data-spectrum-doublets

> In the data-derived weight matrix the leading eigenvalues are approximately real and come in
> visible but inexact doublets.

In [ ]:
t = time.time()
w, V = np.linalg.eig(J)
o = np.argsort(w.real)[::-1]
w, V = w[o], V[:, o]
lead = w[:10]
print(f"eigendecomposition in {time.time()-t:.0f} s")
uni = np.abs(V[:, :10].sum(0)) / (np.sqrt(N) * np.linalg.norm(V[:, :10], axis=0))
print("leading 10 eigenvalues:", [f"{z.real:.4f}{z.imag:+.4f}i" for z in lead])
print("overlap of each eigenvector with the uniform vector:", uni.round(3))
real_ok = np.all(np.abs(lead.imag) < 0.1 * lead.real)
print("approximately real (|Im| < 0.1 Re for all 10):", real_ok, "| max |Im|/Re", (np.abs(lead.imag) / lead.real).max().round(4))
u = int(np.argmax(uni))
rest = np.delete(lead.real, u)
pairs = [(rest[2 * p], rest[2 * p + 1]) for p in range(4)]
within = [a_ - b_ for a_, b_ in pairs]
to_next = [rest[2 * p + 1] - rest[2 * p + 2] for p in range(4)]
ok_pairs = [wi < tn for wi, tn in zip(within, to_next)]
print(f"set aside eigenvalue {u+1} ({lead[u].real:.4f}), the most uniform eigenvector")
print("pairs (real parts):", [tuple(np.round(p, 4)) for p in pairs])
print("gap within each pair:", np.round(within, 4), "| gap to the next pair:", np.round(to_next, 4), "| within < next:", ok_pairs)
print("doublets (>= 3 of 4):", sum(ok_pairs) >= 3, "| not exact (some pair differs by more than 1e-6 relative):", any(abs(wi) > 1e-6 * abs(p[0]) for wi, p in zip(within, pairs)))
# variant: pairs taken without setting any eigenvalue aside
pairs0 = [(lead.real[2 * p], lead.real[2 * p + 1]) for p in range(4)]
wi0 = [a_ - b_ for a_, b_ in pairs0]; tn0 = [lead.real[2 * p + 1] - lead.real[2 * p + 2] for p in range(4)]
print("variant, no eigenvalue set aside: within", np.round(wi0, 4), "next", np.round(tn0, 4), "| within < next:", [bool(a_ < b_) for a_, b_ in zip(wi0, tn0)], f"({sum(a_ < b_ for a_, b_ in zip(wi0, tn0))} of 4)")
# variants: the constant mode is not unique (eigenvectors 5 to 7 tie in their overlap with the
# uniform vector), so each of the three is set aside in turn; and pairs that are complex
# conjugates (equal real parts by construction) are marked
conj = lambda a_, b_: abs(a_.imag) > 1e-9 and np.isclose(a_, np.conj(b_))
for u_ in (4, 5, 6):
    rest_c = np.delete(lead, u_)
    pr = [(rest_c[2 * p], rest_c[2 * p + 1]) for p in range(4)]
    wi = [a_.real - b_.real for a_, b_ in pr]; tn = [rest_c[2 * p + 1].real - rest_c[2 * p + 2].real for p in range(4)]
    ok = [w_ < n_ for w_, n_ in zip(wi, tn)]; cj = [conj(a_, b_) for a_, b_ in pr]
    print(f"variant, eigenvalue {u_ + 1} set aside: within < next {ok}, complex-conjugate pairs {cj}; pairs passing that are not conjugates {sum(o_ and not c_ for o_, c_ in zip(ok, cj))} of 4")
print("eigenvalues: real range", w.real.min().round(3), w.real.max().round(3), "| imag range", w.imag.min().round(3), w.imag.max().round(3))

**Result: reproduced.** The leading 10 eigenvalues have imaginary parts at most 6.8% of their real
parts. After setting aside the fifth (1.0894, whose eigenvector is closest to uniform), the four
leading pairs have gaps of 0.027, 0.000, 0.0002 and 0.000 within, each smaller than the gap to the
next pair (0.139, 0.183, 0.036, 0.0002); the first pair (1.4384 and 1.4113) is not exact. The
evidence is weaker than the count: pairs 2 and 4 are complex conjugates, equal in real part by
construction, so 2 of the 4 passing pairs are not conjugates; eigenvectors 5 to 7 tie in their
overlap with the uniform vector (0.041), and setting aside the sixth or the seventh in place of the
fifth also passes 4 of 4 (2 and 1 of them not conjugates).

## figs11b-data-network-global-stability

> In the data-derived network, initial conditions far from the manifold converge to points on
> it, with no spurious attractors, and initial conditions close to the origin converge to one or
> a few states on the ring.

As the data-derived part of `compute_spurious_analysis` (`spurious_fixed_points.ipynb`): 34
initial states i.i.d. Gaussian with zero mean and standard deviation 1, 10, 50, 100 or 1,000 per
neuron, softplus network with c 1, step tau/10 = 5 ms, 30 s. The final firing rates are compared
with the distance to the manifold reached at 20 s in fig2ef, the largest over its 100
trajectories. The final distances sit at the network's resolution (about 1e-3), and the fig2ef
run uses a 1-ms step where this one uses 5 ms, which leaves the final states slightly farther
from the targets; so the threshold is taken from the fig2ef protocol (its seed and its 100 noisy
starts) run with the 5-ms step of these runs (assumption). The largest and median fig2ef
distances with the 1-ms step are shown as variants. The final manifold angles at standard
deviation 1 are counted as states with the merging readings of fig2gi (width 2 pi / 50).

In [ ]:
INIT_D = {}
def run_s11b(sd):
    rng = np.random.default_rng(sd)
    out = {}
    for sv in (1, 10, 50, 100, 1000):
        x0 = rng.standard_normal((N, 34)) * sv
        INIT_D.setdefault(sd, {})[sv] = np.median(H.nearest_target(Phi32, H.nonlin(x0.astype(np.float32)), Phi_sq)[1])
        xf = H.simulate(J, x0, T=30.0, dt=0.005, c=1.0)
        idx, dist = H.nearest_target(Phi32, H.nonlin(xf), Phi_sq)
        out[sv] = (idx, dist, np.isfinite(xf).all())
    return out
t = time.time()
print(f"thresholds from fig2ef at 20 s (1-ms step, as fig2ef): largest {D20_MAX:.5f}, median {D20_MED:.5f}")
# the reference on the same footing as the runs below: the fig2ef protocol (its seed, its 100 noisy
# manifold starts) with the 5-ms step of spurious_fixed_points.ipynb, distance at 20 s and at the end
rng_ref = np.random.default_rng(H.seed("fig2ef-data-network-quasicontinuous"))
x0_ref = X[::100].T + rng_ref.standard_normal((N, 100)) * 3.0
ref5 = {}
def on_save_ref(tt, x):
    if abs(tt - 20.0) < 1e-6:
        ref5[20] = H.nearest_target(Phi32, H.nonlin(x), Phi_sq)[1]
xf_ref = H.simulate(J, x0_ref, T=30.0, dt=0.005, c=1.0, save_every=0.1, on_save=on_save_ref)
ref5["end"] = H.nearest_target(Phi32, H.nonlin(xf_ref), Phi_sq)[1]
D20_MAX5 = ref5[20].max()
print(f"fig2ef protocol with the 5-ms step: distance at 20 s median {np.median(ref5[20]):.5f}, largest {D20_MAX5:.5f}; at the end (29.995 s) median {np.median(ref5['end']):.5f}, largest {ref5['end'].max():.5f}")
for k in range(5):
    sd = H.seed("figs11b-data-network-global-stability") + 1000 * k
    out = run_s11b(sd)
    line = []
    for sv, (idx, dist, fin) in out.items():
        line.append(f"sd {sv}: finite {fin}, median dist {np.median(dist):.5f}, max {dist.max():.5f}, within largest (5-ms reference) {np.mean(dist <= D20_MAX5):.2f}, "
                    f"within largest (1-ms fig2ef) {np.mean(dist <= D20_MAX):.2f}, within median {np.mean(dist <= D20_MED):.2f}")
    cs = H.count_states(out[1][0], X.shape[0], X.shape[0] / 50)
    print(f"seed {sd}: " + "; ".join(line) + f"; sd 1 final angles as states: {cs}")
    print(f"   at least 95% within the 5-ms reference at every standard deviation: {all(np.mean(o[1] <= D20_MAX5) >= 0.95 for o in out.values())}; "
          f"within the 1-ms one: {all(np.mean(o[1] <= D20_MAX) >= 0.95 for o in out.values())}", flush=True)
    if k == 0:
        out0 = out
print(f"time {time.time()-t:.0f} s")
print("median initial distance to the manifold by standard deviation, claim seed:", {k_: round(float(v_), 3) for k_, v_ in INIT_D[H.seed("figs11b-data-network-global-stability")].items()})
print("sd 1 final manifold angles (rad), claim seed:", np.sort(2 * np.pi * out0[1][0] / X.shape[0]).round(2))

**Result: reproduced (resting on an assumption).** From median starting distances of 1.6 to 700,
every final state ends within 0.0016 of the manifold (final medians 0.00047 to 0.00085, largest
0.00154), with none elsewhere, and at standard deviation 1 all 34 final angles fall in one state
(1.69 to 1.70 rad), under all five seeds. Against the fig2ef protocol run with the same 5-ms step
(largest distance at 20 s 0.00141), 97% to 100% of final states are within at every standard
deviation, under all five seeds; the largest final distances (0.00147, 0.00153 and 0.00154) are
just above that reference. Against the 1-ms fig2ef threshold (0.00132), the variant, 91% to 100%
are within, below 95% at standard deviations 10, 50 and 1,000 under the claim's seed (0.94, 0.91,
0.94); the final distances sit at the network's resolution, about 1e-3, where the step of the
reference decides the test.

## Summary

| claim | outcome |
|---|---|
| fig2b-sorted-weights-mexican-hat | reproduced |
| fig2ef-data-network-quasicontinuous | reproduced |
| fig2gi-circulant-noise-collapses | partially reproduced |
| fig2jk-velocity-integration | reproduced |
| figs14-data-spectrum-doublets | reproduced |
| figs11b-data-network-global-stability | reproduced (assumed) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

Differences from the paper's method. Simulations in numpy float32 in place of torch float32;
the distance to the manifold is read every 100 ms (the paper saves every 10 ms); random draws
come from each claim's seed; the threshold of figs11b comes from the fig2ef protocol run with the
5-ms step of the runs it judges (assumption). The network built from the dataset reproduces the paper's printed
reconstruction error (1.7152e-08) to five digits.

In [ ]:
print(f"total time {time.time()-T0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.2f} GB")